In [ ]:
%matplotlib widget

In [ ]:
import flammkuchen as fl
import numpy as np
from pathlib import Path
import tifffile as tiff
from split_dataset import SplitDataset

In [ ]:
import matplotlib.pyplot as plt
from fimpy.pipeline.roi_extraction import extract_traces

In [ ]:
from scipy.stats import zscore
from bouterin.plots.stimulus_log_plot import get_paint_function
from fimpylab.core.twop_experiment import TwoPExperiment


In [ ]:
master = Path(r"Z:\Hagar\development\gad1b6s 2p\selected planes")
fish_list = list(master.glob("*_f*"))
fish = fish_list[9]
fish_list

In [ ]:
###  extract traces for all fish
for fish in fish_list:
    print(fish)
    try:
        if not (fish / "traces.h5").exists():
            print("Extracting traces")
            rois = fl.load(fish / 'manual_rois.h5', '/labels_rois')
            aligned = SplitDataset(fish / "aligned")[:,:,:,:]

            n_rois = np.max(rois)
            n_t = np.shape(aligned)[0]
            traces = np.zeros((n_rois, n_t))
            norm_traces = np.zeros((n_rois, n_t))

            t_imaging = np.arange(0, np.shape(aligned)[0]) // 3

            for roi in range(1, n_rois+1):
                curr_roi = np.where(rois == roi)
                #trace = aligned[:, i, curr_roi[1], curr_roi[2]]
                try:
                    z = curr_roi[0][1]
                    center_x = np.nanmean(curr_roi[2])
                    center_y = np.nanmean(curr_roi[1])

                    num_pix = curr_roi[1]
                    traces[roi-1] = np.nanmean(aligned[:, z, curr_roi[1], curr_roi[2]], axis=1)
                    norm_traces[roi-1] = zscore(traces[roi-1])

                except:
                    print("no rois")

            d = {'traces': traces,
                'norm_traces': norm_traces}
            fl.save(fish / 'traces.h5', d)
        else:
            print("Traces already extracted")
            
    except:
        print("Stupid fish")

In [ ]:
rois = fl.load(fish / 'manual_rois.h5', '/labels_rois')
anatomy = tiff.imread(fish / 'anatomy.tif')

In [ ]:
try:
    traces = fl.load(fish / 'traces.h5', '/norm_traces')

except:    
    aligned = SplitDataset(fish / "aligned")[:,:,:,:]

In [ ]:
num_planes = np.shape(anatomy)[0]
print("num planes: ", num_planes)

In [ ]:
exp = TwoPExperiment(path=fish)

stimulus_log = exp.load_session_log(log_name='stimulus_log', session_idx=0)
stim_value, t_values = get_paint_function(stimulus_log, 'E0040_motions_cardinal')
stim_value = stim_value / 255
num_stim = np.shape(stim_value)[0]

In [ ]:
if nun_planes > 1:
    
    fig, axs = plt.subplots(num_planes, 2, figsize=(8, 12), gridspec_kw={'width_ratios': [1,4]})
    fig.subplots_adjust(top=0.99, bottom=0.1, left=0.1, right=0.9, wspace=0.1)
    for i in range(num_planes):
        axs[i, 0].imshow(np.rot90(anatomy[i], 0), cmap="gray_r", vmin=0, vmax=1)
        axs[i, 0].axis('off')
        axs[i, 0].invert_yaxis()

        axs[i, 1].axis('off')
        for stim in range(num_stim):
            axs[i, 1].axvspan(
                t_values[stim, 0],
                t_values[stim, 1],
                facecolor=[stim_value[stim, 0], stim_value[stim, 1], stim_value[stim, 2]],
                alpha=0.5,
            )
else:
    
    fig, axs = plt.subplots(num_planes, 2, figsize=(8, 4), gridspec_kw={'width_ratios': [1,4]})
    fig.subplots_adjust(top=0.99, bottom=0.1, left=0.1, right=0.9, wspace=0.1)
    for i in range(num_planes):
        axs[0].imshow(np.rot90(anatomy[i], 0), cmap="gray_r", vmin=0, vmax=1)
        axs[0].axis('off')
        axs[0].invert_yaxis()

        axs[1].axis('off')
        for stim in range(num_stim):
            axs[1].axvspan(
                t_values[stim, 0],
                t_values[stim, 1],
                facecolor=[stim_value[stim, 0], stim_value[stim, 1], stim_value[stim, 2]],
                alpha=0.5,
            )
    

In [ ]:
n_rois = np.max(rois)
n_t = np.shape(aligned)[0]

traces = np.zeros((n_rois, n_t))
norm_traces = np.zeros((n_rois, n_t))

In [ ]:
t_imaging = np.arange(0, np.shape(aligned)[0]) // 3

if nun_planes > 1:
    
    for roi in range(1, n_rois+1):
        curr_roi = np.where(rois == roi)
        #trace = aligned[:, i, curr_roi[1], curr_roi[2]]
        try:
            z = curr_roi[0][1]
            center_x = np.nanmean(curr_roi[2])
            center_y = np.nanmean(curr_roi[1])
            axs[z, 0].scatter(center_x, center_y, s=2)

            num_pix = curr_roi[1]
            traces[roi-1] = np.nanmean(aligned[:, z, curr_roi[1], curr_roi[2]], axis=1)
            norm_traces[roi-1] = zscore(traces[roi-1])

            axs[z, 1].plot(t_imaging, norm_traces[roi-1] + (5 * (roi)), linewidth=0.5)
        except:
            print("no rois")
else:
    for roi in range(1, n_rois+1):
        curr_roi = np.where(rois == roi)
        #trace = aligned[:, i, curr_roi[1], curr_roi[2]]
        try:
            z = curr_roi[0][1]
            center_x = np.nanmean(curr_roi[2])
            center_y = np.nanmean(curr_roi[1])
            axs[0].scatter(center_x, center_y, s=2)

            num_pix = curr_roi[1]
            traces[roi-1] = np.nanmean(aligned[:, z, curr_roi[1], curr_roi[2]], axis=1)
            norm_traces[roi-1] = zscore(traces[roi-1])

            axs[1].plot(t_imaging, norm_traces[roi-1] + (5 * (roi)), linewidth=0.5)
        except:
            print("no rois")


    

In [ ]:
d = {'traces': traces,
    'norm_traces': norm_traces}
fl.save(fish / 'traces.h5', d)

In [ ]:
if nun_planes > 1:
    fig2, axs2 = plt.subplots(num_planes, 2, figsize=(8, 12), gridspec_kw={'width_ratios': [1,4]})
    fig2.subplots_adjust(top=0.99, bottom=0.1, left=0.1, right=0.9, wspace=0.1)
    for i in range(num_planes):
        axs2[i, 0].imshow(np.rot90(anatomy[i], 0), cmap="gray_r", vmin=0, vmax=1)
        axs2[i, 0].axis('off')
        axs2[i, 0].invert_yaxis()

        axs2[i, 1].axis('off')
        for stim in range(num_stim//3):
            axs2[i, 1].axvspan(
                t_values[stim, 0],
                t_values[stim, 1],
                facecolor=[stim_value[stim, 0], stim_value[stim, 1], stim_value[stim, 2]],
                alpha=0.5,
            )
else:
    fig2, axs2 = plt.subplots(num_planes, 2, figsize=(8, 4), gridspec_kw={'width_ratios': [1,4]})
    fig2.subplots_adjust(top=0.99, bottom=0.1, left=0.1, right=0.9, wspace=0.1)
    for i in range(num_planes):
        axs2[0].imshow(np.rot90(anatomy[i], 0), cmap="gray_r", vmin=0, vmax=1)
        axs2[0].axis('off')
        axs2[0].invert_yaxis()

        axs2[1].axis('off')
        for stim in range(num_stim//3):
            axs2[1].axvspan(
                t_values[stim, 0],
                t_values[stim, 1],
                facecolor=[stim_value[stim, 0], stim_value[stim, 1], stim_value[stim, 2]],
                alpha=0.5,
            )
    

In [ ]:
n_rois = np.max(rois)

t_imaging = np.arange(0, np.shape(aligned)[0]) // 3

if nun_planes > 1:    
    for roi in range(1, n_rois+1):
        curr_roi = np.where(rois == roi)
        #trace = aligned[:, i, curr_roi[1], curr_roi[2]]

        try:
            z = curr_roi[0][1]
            center_x = np.nanmean(curr_roi[2])
            center_y = np.nanmean(curr_roi[1])
            axs2[z, 0].scatter(center_x, center_y, s=2)

            num_pix = curr_roi[1]
            trace = zscore(np.nanmean(aligned[:, z, curr_roi[1], curr_roi[2]], axis=1))
            trace_reshape = np.reshape(trace[:768*3], (3, np.shape(aligned)[0]//3))
            trace_avg = np.nanmean(trace_reshape, axis=0)

            axs2[z, 1].plot(t_imaging[:768], trace_avg + (5 * (roi)), linewidth=0.5)
        except:
            print("no roi")
else:
    for roi in range(1, n_rois+1):
        curr_roi = np.where(rois == roi)
        #trace = aligned[:, i, curr_roi[1], curr_roi[2]]

        try:
            z = curr_roi[0][1]
            center_x = np.nanmean(curr_roi[2])
            center_y = np.nanmean(curr_roi[1])
            axs2[0].scatter(center_x, center_y, s=2)

            num_pix = curr_roi[1]
            trace = zscore(np.nanmean(aligned[:, z, curr_roi[1], curr_roi[2]], axis=1))
            trace_reshape = np.reshape(trace[:768*3], (3, np.shape(aligned)[0]//3))
            trace_avg = np.nanmean(trace_reshape, axis=0)

            axs2[1].plot(t_imaging[:768], trace_avg + (5 * (roi)), linewidth=0.5)
        except:
            print("no roi")

    

In [ ]:
fig.savefig(fish / "manual rois and traces.pdf", dpi=300)
fig.savefig(fish / "manual rois and traces.jpg", dpi=300)

In [ ]:
fig2.savefig(fish / "manual rois and traces avg.pdf", dpi=300)
fig2.savefig(fish / "manual rois and traces avg.jpg", dpi=300)